<a href="https://colab.research.google.com/github/mahmoadalmasry2020-cyber/Fine-tuning-a-model-with-the-Trainer-API/blob/main/Fine_tuning_a_model_with_the_Trainer_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
!pip install -U datasets huggingface_hub
!pip install evaluate


In [3]:
!pip uninstall -y torchaudio torchvision

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

datasets =load_dataset("nyu-mll/glue","mrpc")
checkpoint= "bert-base-uncased"
tokenizer= AutoTokenizer.from_pretrained(checkpoint)

def tokenized_function(example):
    return tokenizer(example["sentence1"],example["sentence2"],truncation=True)

tokenized_dataset=datasets.map(tokenized_function,batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)


In [10]:
from transformers import TrainingArguments

training_arg=TrainingArguments("test-trainer",
                               eval_strategy="epoch",
                               per_device_train_batch_size=4,
                               gradient_accumulation_steps=4,
                               learning_rate=2e-5,
                               lr_scheduler_type="cosine",
                               )

In [6]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
import evaluate
import numpy as np

def compute_metrics(eval_preds):
    metrics=evaluate.load("glue","mrpc")
    logits,labels=eval_preds
    predictions=np.argmax(logits,axis=-1)
    return metrics.compute(predictions=predictions, references=labels)

In [8]:
from transformers import Trainer

trainer=Trainer(
    model,
    training_arg,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.432867,0.803922,0.869707
2,No log,0.391751,0.830882,0.880829
3,1.691179,0.435468,0.828431,0.880137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=690, training_loss=1.4168558535368547, metrics={'train_runtime': 269.6743, 'train_samples_per_second': 40.805, 'train_steps_per_second': 2.559, 'total_flos': 377531475559680.0, 'train_loss': 1.4168558535368547, 'epoch': 3.0})